# Minimal Vision Transformer (ViT): Patch Embedding + Training on CIFAR-10

This notebook trains a **minimal Vision Transformer** (Dosovitskiy et al.,
2021, "An Image Is Worth 16x16 Words: Transformers for Image Recognition
at Scale") on CIFAR-10. It is the **train** notebook of a train/evaluate
pair: it trains on GPU and saves a checkpoint + log to Drive;
`vit_evaluate.ipynb` loads that checkpoint afterwards to run all
visualizations, so changing a plot later never requires retraining.

Patch + position embedding (Eq. 1) and the Pre-LN encoder layer (Eq. 2-3):

$$z_0 = [x_{class}; x_p^1 E; x_p^2 E; \dots; x_p^N E] + E_{pos}$$
$$z'_l = \text{MSA}(\text{LN}(z_{l-1})) + z_{l-1}, \qquad z_l = \text{MLP}(\text{LN}(z'_l)) + z'_l$$

| symbol | code variable | shape | meaning |
|---|---|---|---|
| image | `images` | `[B,C,H,W]` | input image batch, C=3, H=W=32 for CIFAR-10 |
| P (patch size) | `patch_size` | scalar | side length of one square patch |
| N (#patches) | `num_patches` | scalar | `(H/P)*(W/P)`, token count contributed by the image |
| E (patch projection) | `patch_embedding.projection.weight` | `[D,P*P*C]` | learned linear patch projection |
| D (embed dim) | `embed_dim` | scalar | Transformer token width |
| $E_{pos}$ | `position_embedding` | `[1,N+1,D]` | learned positional embedding (+1 for CLS) |
| encoder output | `x` | `[B,N+1,D]` | shape-preserving stack of `ViTEncoderBlockPreLN` |
| logits | `logits` | `[B,10]` | CIFAR-10 class scores |

**Independent of the Vaswani-Transformer exercise**: `code/transformers/vit_patch_embedding.py`
implements its own `MultiHeadSelfAttention`, `PositionwiseFeedForward`,
and `ViTEncoderBlockPreLN` (Eq. 2-3's Pre-LN wrapping), rather than
importing `TransformerEncoderBlock` from `transformer_block.py` (which
implements the original Vaswani et al., 2017 Post-LN wrapping for a
separate exercise). This keeps the two exercises fully decoupled -- a
change to one can never affect the other -- while making this ViT
faithful to Eq. 2-3 rather than merely shape-compatible with it.

Core code lives in `code/transformers/vit_patch_embedding.py`
(`VisionTransformerClassifier`) and `code/transformers/vit_train.py`
(training loop, CIFAR-10 loading, checkpointing). Unit-checked with
`python code/transformers/vit_patch_embedding.py --smoke-test` and
`python code/transformers/vit_train.py --smoke-test` (tiny synthetic
data, no download, run locally). This notebook is a thin orchestration
layer; real training happens here, on a Colab GPU.

## 1. Check GPU

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected -- Runtime > Change runtime type > GPU")

## 2. Clone/pull repository

ViT training code lives entirely in `code/transformers/vit_patch_embedding.py`
and `vit_train.py` -- these two files are self-contained (no dependency
on `transformer_block.py`), not duplicated in this notebook. Replace
`REPO_URL` with this repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code", "transformers"))

## 3. Mount Google Drive and choose configuration

CIFAR-10 is cached on Drive (`DATA_ROOT`) so it is downloaded once, not
every session; checkpoints and the training log are also written to
Drive so they survive Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/thesis_data/cifar10"
CHECKPOINT_PATH = "/content/drive/MyDrive/thesis_data/vit_cifar10/checkpoint.pt"
LOG_PATH = "/content/drive/MyDrive/thesis_data/vit_cifar10/train_log.jsonl"

PATCH_SIZE = 4
EMBED_DIM = 64
NUM_HEADS = 4
NUM_LAYERS = 3
FFN_HIDDEN_DIM = 128
BATCH_SIZE = 128
LR = 3e-4
EPOCHS = 30
SEED = 0
RESUME = False  # set True to continue from CHECKPOINT_PATH after a disconnect

## 4. Launch training

Runs the same `train()` function used by
`python code/transformers/vit_train.py` on the command line, with the
configuration above. Trains on GPU if available. The train/test split is
CIFAR-10's own predefined files (not a random re-split); test
performance is evaluated every epoch but never used for gradient
updates. A checkpoint (model + optimizer + epoch + config) is saved to
`CHECKPOINT_PATH` after every epoch, so this cell can be safely re-run
with `RESUME = True` above if the session disconnects.

In [ ]:
import argparse
from vit_train import train

args = argparse.Namespace(
    smoke_test=False,
    patch_size=PATCH_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ffn_hidden_dim=FFN_HIDDEN_DIM,
    num_train=None,  # unused: only the real CIFAR-10 split is used when smoke_test=False
    num_test=None,
    batch_size=BATCH_SIZE,
    lr=LR,
    epochs=EPOCHS,
    seed=SEED,
    data_root=DATA_ROOT,
    checkpoint_path=CHECKPOINT_PATH,
    log_path=LOG_PATH,
    resume=RESUME,
)

train(args)

## 5. Next step

Open `vit_evaluate.ipynb` with `CHECKPOINT_PATH`, `LOG_PATH`, and
`DATA_ROOT` set to the same paths as above, to compute test accuracy and
generate all visualizations (training curves, confusion matrix, sample
predictions, CLS attention maps, position-embedding similarity) without
retraining.